In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import nltk
nltk.download("stopwords")
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
stop_words = set(stopwords.words("english"))
from nltk.stem import PorterStemmer
from nltk.stem.wordnet import WordNetLemmatizer
lemma = WordNetLemmatizer()
ps = PorterStemmer()
import re


from scipy.spatial import distance
from scipy.spatial import minkowski_distance
from scipy.spatial.distance import cosine




[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/sophie/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [3]:
#Electronics Dataset:

import numpy as np
import pandas as pd
fileElectr='amazon_reviews_us_Electronics_v1_00.tsv'
df=pd.read_csv(fileElectr, sep="\t", header=0, on_bad_lines='skip')
df=df.dropna(subset=['review_headline', 'review_body', 'star_rating'])

In [4]:
df.star_rating.value_counts()

5    1779304
4     536398
1     357800
3     238378
2     179025
Name: star_rating, dtype: int64

In [13]:
from transformers import pipeline


    
    
def emotion_roberta(df_sample, column_name):
    classifier = pipeline("text-classification", model="j-hartmann/emotion-english-distilroberta-base", return_all_scores=True)

    for i in range (0, len(df_sample[column_name])):

        #processed text
        text=df_sample[column_name][i]  

        if len(text) > 512:
            text = text[:512]

        prediction = classifier(text )

        for j in range(0,7):  #loop over the  emotions:
            df_sample.loc[i, ("roberta_"+column_name+prediction[0][j]['label'])]=prediction[0][j]['score'] #BP=body processed


        if i%1000==0:
           print ("\n emotion_roberta:   We are at i=", str(i))    
            
            
    return df_sample




In [6]:
def text_process2(reviews, column_name):  #input is the dataframe
    for i  in range(0, reviews[column_name].count()):
       review_body=reviews.loc[i, (column_name)]  #tokens= word_tokenize(df_sample.loc[1, ('review_body')])
       review_body=re.sub('<br\s?\/>|<br>', " ", review_body)  #remove the br
       tokens= word_tokenize(review_body)
       tokens = [w.lower()  for w in tokens ]
       #tokens = [w for w in tokens if not w in stop_words]
       tokens = [w for w in tokens if w.isalpha()] #remove non alphabetic items like like 5 or ;
       tokens = [lemma.lemmatize(w) for w in tokens]
       # tokens = [ps.stem(w) for w in tokens]
       column_name_out=column_name+"_processed"
       reviews.loc[i, (column_name_out)]=' '.join(tokens)
       
       if i%10000==0:
           print ("\n text_process:   We are at i=", str(i))
       
    return reviews





In [14]:
n_samples=10000

N_rewiews=df.loc[df['star_rating'] == 1].sample(n_samples, replace=False, random_state=1900)
N_rewiew2=df.loc[df['star_rating'] == 5].sample(n_samples, replace=False, random_state=1900)


samplesize=n_samples*2

N_rewiews=N_rewiews.append(N_rewiew2)



/var/folders/ds/k83592y50w34wqwchx8qldph0000gn/T/ipykernel_5253/3763742897.py:9: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  N_rewiews=N_rewiews.append(N_rewiew2)


In [15]:
N_rewiews=N_rewiews.reset_index()

In [16]:
#N_rewiews=text_process2(N_rewiews,'review_body')
N_rewiews=text_process2(N_rewiews,'review_headline')







 text_process:   We are at i= 0

 text_process:   We are at i= 10000


In [17]:
emotion_roberta(N_rewiews, 'review_body')
#emotion_distilbert(N_rewiews, 'review_body')
#
#emotion_roberta(N_rewiews, 'review_body_processed')
#
#emotion_distilbert(N_rewiews, 'review_body_processed')




 emotion_roberta:   We are at i= 0

 emotion_roberta:   We are at i= 1000

 emotion_roberta:   We are at i= 2000

 emotion_roberta:   We are at i= 3000

 emotion_roberta:   We are at i= 4000

 emotion_roberta:   We are at i= 5000

 emotion_roberta:   We are at i= 6000

 emotion_roberta:   We are at i= 7000

 emotion_roberta:   We are at i= 8000

 emotion_roberta:   We are at i= 9000

 emotion_roberta:   We are at i= 10000

 emotion_roberta:   We are at i= 11000

 emotion_roberta:   We are at i= 12000

 emotion_roberta:   We are at i= 13000

 emotion_roberta:   We are at i= 14000

 emotion_roberta:   We are at i= 15000

 emotion_roberta:   We are at i= 16000

 emotion_roberta:   We are at i= 17000

 emotion_roberta:   We are at i= 18000

 emotion_roberta:   We are at i= 19000


,index,marketplace,customer_id,review_id,product_id,product_parent,product_title,product_category,star_rating,helpful_votes,...,review_body,review_date,review_headline_processed,roberta_review_bodyanger,roberta_review_bodydisgust,roberta_review_bodyfear,roberta_review_bodyjoy,roberta_review_bodyneutral,roberta_review_bodysadness,roberta_review_bodysurprise
0,1677873,US,18585476,R31LJGNJRWGRS,B003ARSOWQ,864558418,Timex T715BW3 Dual Alarm Clock Radio (Black),Electronics,1,0,...,"I have had the product for just under a month,...",2013-12-14,doe it keep time not really,0.148382,0.261074,0.011709,0.001612,0.522428,0.038445,0.016350
1,1856468,US,51075252,R35YF0DWJE87A4,B0044WS7KK,471031907,"Aerial7 Perisher - Black - black, one size",Electronics,1,0,...,The speaker quality reminds you of a 1960's tr...,2013-08-12,very poor sound,0.002942,0.005885,0.005614,0.006947,0.032595,0.002518,0.943499
2,600712,US,15700552,R2PQJHUI1SF24E,B00INO6JX2,703104763,Samsung SSG-5150GB 3D Active Glasses,Electronics,1,1,...,Did not work,2015-02-24,wrong item,0.015367,0.015098,0.006451,0.001496,0.121806,0.820625,0.019158
3,2871105,US,26397372,R1B30LZ8X5V07Q,B00008VSK5,50332,Acoustic Research MS805 Adaptatip Flex Pin,Electronics,1,4,...,This flex pin set requires an additional part:...,2008-12-22,flex pin set incomplete not worth it,0.260976,0.092722,0.015083,0.002443,0.487012,0.094730,0.047033
4,1381712,US,20903669,R2B9JFV8C5AHX,B007N16IYG,623454097,Panasonic Deep Base Ergo-Fit Inner Ear Earbud ...,Electronics,1,1,...,These are not decent. This earphone set is spl...,2014-05-16,these are not what you think much regret,0.052197,0.902236,0.003806,0.000668,0.018588,0.015990,0.006515
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19995,2273697,US,30170180,R1JG9WFRJH44NB,B003XO57VM,846711659,Philips Lighting Sony KDF-60XS955 KDF60XS955 L...,Electronics,5,2,...,"Came promptly, easy to install (I'm not handy)...",2012-11-11,oem quality easy install,0.017919,0.004830,0.005972,0.095326,0.377220,0.005577,0.493156
19996,1200249,US,44475306,R2MMLE26RF3WC2,B00358VSI2,599267792,Coby Digital Active Noise-Canceling Stereo Hea...,Electronics,5,0,...,I wanted noise cancelling headphones to use wh...,2014-08-12,nice headphone,0.007445,0.013562,0.007493,0.078569,0.851484,0.025257,0.016191
19997,852624,US,46755668,R1X86QEU74V7LK,B00BN0N0N0,21856130,Sony MDRAS200 Active Sports Headphones,Electronics,5,0,...,"Sound is excellent, they stay put running, wal...",2014-12-20,five star,0.004954,0.013535,0.002225,0.279442,0.657715,0.031152,0.010976
19998,757939,US,40385685,R2JLSZ9EJHV4XS,B0077QMHVU,311150041,WILSON ANTENNAS Noise Canceling CB Microphone ...,Electronics,5,0,...,Excellent ممتازه,2015-01-12,five star,0.019810,0.023431,0.007185,0.047440,0.887383,0.006005,0.008747


In [18]:
emotion_roberta(N_rewiews, 'review_headline')
#emotion_distilbert(N_rewiews, 'review_headline')


emotion_roberta(N_rewiews, 'review_headline_processed')
#emotion_distilbert(N_rewiews, 'review_headline_processed')





 emotion_roberta:   We are at i= 0

 emotion_roberta:   We are at i= 1000

 emotion_roberta:   We are at i= 2000

 emotion_roberta:   We are at i= 3000

 emotion_roberta:   We are at i= 4000

 emotion_roberta:   We are at i= 5000

 emotion_roberta:   We are at i= 6000

 emotion_roberta:   We are at i= 7000

 emotion_roberta:   We are at i= 8000

 emotion_roberta:   We are at i= 9000

 emotion_roberta:   We are at i= 10000

 emotion_roberta:   We are at i= 11000

 emotion_roberta:   We are at i= 12000

 emotion_roberta:   We are at i= 13000

 emotion_roberta:   We are at i= 14000

 emotion_roberta:   We are at i= 15000

 emotion_roberta:   We are at i= 16000

 emotion_roberta:   We are at i= 17000

 emotion_roberta:   We are at i= 18000

 emotion_roberta:   We are at i= 19000

 emotion_roberta:   We are at i= 0

 emotion_roberta:   We are at i= 1000

 emotion_roberta:   We are at i= 2000

 emotion_roberta:   We are at i= 3000

 emotion_roberta:   We are at i= 4000

 emotion_roberta:   

,index,marketplace,customer_id,review_id,product_id,product_parent,product_title,product_category,star_rating,helpful_votes,...,roberta_review_headlineneutral,roberta_review_headlinesadness,roberta_review_headlinesurprise,roberta_review_headline_processedanger,roberta_review_headline_processeddisgust,roberta_review_headline_processedfear,roberta_review_headline_processedjoy,roberta_review_headline_processedneutral,roberta_review_headline_processedsadness,roberta_review_headline_processedsurprise
0,1677873,US,18585476,R31LJGNJRWGRS,B003ARSOWQ,864558418,Timex T715BW3 Dual Alarm Clock Radio (Black),Electronics,1,0,...,0.683399,0.037004,0.196546,0.010669,0.017760,0.007133,0.007847,0.839019,0.079229,0.038343
1,1856468,US,51075252,R35YF0DWJE87A4,B0044WS7KK,471031907,"Aerial7 Perisher - Black - black, one size",Electronics,1,0,...,0.407534,0.376775,0.032329,0.022509,0.095116,0.019774,0.002500,0.122202,0.713169,0.024730
2,600712,US,15700552,R2PQJHUI1SF24E,B00INO6JX2,703104763,Samsung SSG-5150GB 3D Active Glasses,Electronics,1,1,...,0.031080,0.012691,0.002935,0.042930,0.037332,0.011042,0.004079,0.698276,0.124367,0.081975
3,2871105,US,26397372,R1B30LZ8X5V07Q,B00008VSK5,50332,Acoustic Research MS805 Adaptatip Flex Pin,Electronics,1,4,...,0.309574,0.564732,0.034902,0.031687,0.021541,0.011392,0.006795,0.329728,0.535445,0.063412
4,1381712,US,20903669,R2B9JFV8C5AHX,B007N16IYG,623454097,Panasonic Deep Base Ergo-Fit Inner Ear Earbud ...,Electronics,1,1,...,0.426625,0.491464,0.024036,0.013662,0.043480,0.015211,0.003589,0.608421,0.278068,0.037570
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19995,2273697,US,30170180,R1JG9WFRJH44NB,B003XO57VM,846711659,Philips Lighting Sony KDF-60XS955 KDF60XS955 L...,Electronics,5,2,...,0.953282,0.003900,0.017110,0.005241,0.002233,0.001612,0.214094,0.685456,0.016548,0.074817
19996,1200249,US,44475306,R2MMLE26RF3WC2,B00358VSI2,599267792,Coby Digital Active Noise-Canceling Stereo Hea...,Electronics,5,0,...,0.800938,0.017631,0.028434,0.004495,0.024509,0.002833,0.151292,0.750572,0.025584,0.040714
19997,852624,US,46755668,R1X86QEU74V7LK,B00BN0N0N0,21856130,Sony MDRAS200 Active Sports Headphones,Electronics,5,0,...,0.874900,0.009004,0.035351,0.017302,0.005385,0.002486,0.472787,0.424965,0.012054,0.065021
19998,757939,US,40385685,R2JLSZ9EJHV4XS,B0077QMHVU,311150041,WILSON ANTENNAS Noise Canceling CB Microphone ...,Electronics,5,0,...,0.874900,0.009004,0.035351,0.017302,0.005385,0.002486,0.472787,0.424965,0.012054,0.065021


In [ ]:
#NOT USED 
#Roberta_Body_Processed=[
#'roberta_review_body_processedanger',
#'roberta_review_body_processeddisgust',
#'roberta_review_body_processedfear',
#'roberta_review_body_processedjoy',
#'roberta_review_body_processedneutral',
#'roberta_review_body_processedsadness',
#'roberta_review_body_processedsurprise' ] 

In [19]:
#DETAILED REVIEW for Roberta

Roberta_Body=['roberta_review_bodyanger',
'roberta_review_bodydisgust',
'roberta_review_bodyfear',
'roberta_review_bodyjoy',
'roberta_review_bodyneutral',
'roberta_review_bodysadness',
'roberta_review_bodysurprise']



Roberta_Head=[
'roberta_review_headlineanger', 
'roberta_review_headlinedisgust',
'roberta_review_headlinefear', 
'roberta_review_headlinejoy',
'roberta_review_headlineneutral', 
'roberta_review_headlinesadness',
'roberta_review_headlinesurprise']


Roberta_Head_Processed=[
'roberta_review_headline_processedanger', 
'roberta_review_headline_processeddisgust',
'roberta_review_headline_processedfear', 
'roberta_review_headline_processedjoy',
'roberta_review_headline_processedneutral', 
'roberta_review_headline_processedsadness',
'roberta_review_headline_processedsurprise'] 





In [27]:

from sklearn.model_selection import train_test_split

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report
from sklearn.svm import SVC
from sklearn.metrics import confusion_matrix





def run_SVC(df_sample_all_test,N_rewiews):
    target_names = ['0 = rating of 1',  '1 = rating of 5'] # 0 = negative, 4 = positive


    X_train, X_test, y_train, y_test = train_test_split(df_sample_all_test, N_rewiews['star_rating'],  random_state=56)

    clf = make_pipeline(StandardScaler(), SVC( C=1000, gamma= 0.001, kernel= 'rbf'))
    clf.fit(X_train, y_train)
    y_pred=clf.predict(X_test)

    clf.score(X_test, y_test)
    y_test.value_counts()
    print(clf.score(X_test, y_test))

    print(classification_report(y_test, y_pred, target_names=target_names, digits=6))

    print(confusion_matrix(y_test, y_pred))




In [21]:
from sklearn.ensemble import AdaBoostClassifier
from sklearn.datasets import make_classification


def run_AdaBoost(df_sample_all_test,N_rewiews):

    X_train, X_test, y_train, y_test = train_test_split(df_sample_all_test, N_rewiews['star_rating'],  random_state=56)

    clf = AdaBoostClassifier(n_estimators=50, random_state=156)
    
    clf.fit(X_train, y_train)
    y_pred=clf.predict(X_test)

    print(clf.score(X_test, y_test))

#    clf.score(X_test, y_test)
    target_names = ['0 = rating of 1',  '1 = rating of 5'] # 0 = negative, 4 = positive


    print(classification_report(y_test, y_pred, target_names=target_names, digits=6))

    print(confusion_matrix(y_test, y_pred))







In [28]:
df_sample_all_test=N_rewiews[ Roberta_Body+Roberta_Head+Roberta_Head_Processed ] 

run_SVC(df_sample_all_test,N_rewiews)


0.931
                 precision    recall  f1-score   support

0 = rating of 1   0.937046  0.924731  0.930848      2511
1 = rating of 5   0.925059  0.937324  0.931151      2489

       accuracy                       0.931000      5000
      macro avg   0.931053  0.931028  0.931000      5000
   weighted avg   0.931079  0.931000  0.930999      5000

[[2322  189]
 [ 156 2333]]


In [29]:
run_AdaBoost(df_sample_all_test,N_rewiews)



0.939
                 precision    recall  f1-score   support

0 = rating of 1   0.944042  0.933891  0.938939      2511
1 = rating of 5   0.934022  0.944154  0.939061      2489

       accuracy                       0.939000      5000
      macro avg   0.939032  0.939023  0.939000      5000
   weighted avg   0.939054  0.939000  0.939000      5000

[[2345  166]
 [ 139 2350]]


In [30]:
#adding embeddings 

In [31]:
def df2emd(word2vec_model, N_rewiews, column_name):
    word2vec_model_embeddings = WordVecVectorizer(word2vec_model)

    word2vec_model_embeddings_ave_one_review_list=[]
    embed_only=pd.DataFrame()
    
    for i in range(0,len(N_rewiews[column_name]) ) : 
    #for i in range(0,len(N_rewiews['review_body_process']) ) : 
        #list_words=[N_rewiews['review_body_process'][i]]
        list_words=[N_rewiews[column_name][i]]

        list_words=check_against_word2vec_model(list_words, word2vec_model)
        word2vec_embeddings_one_review=word2vec_model_embeddings.transform(list_words)
        word2vec_model_embeddings_ave_one_review_list.append(word2vec_embeddings_one_review)

    embed_only=pd.DataFrame(np.concatenate(word2vec_model_embeddings_ave_one_review_list))
    
    return embed_only

In [32]:

class WordVecVectorizer(object):
    def __init__(self, word2vec_model):
        self.word2vec_model = word2vec_model
        self.dim = 300
    def transform(self, X):
        return np.array([
            np.mean([self.word2vec_model[w] for w in texts.split() if w in self.word2vec_model]
                    or [np.zeros(self.dim)], axis=0)
            for texts in X
        ])




def check_against_word2vec_model(list_topics, word2vec_model):
    for i  in range(0, len(list_topics) ):
       tokens= word_tokenize(list_topics[i])
       tokens = [w for w in tokens if w in word2vec_model.key_to_index ]
       list_topics[i]=' '.join(tokens)
       return list_topics







In [33]:
import gensim


file_embeddings_fast='crawl-300d-2M.vec'


word2vec_model_fast = gensim.models.KeyedVectors.load_word2vec_format(file_embeddings_fast) 
print(word2vec_model_fast.vector_size)





300


In [34]:
embed_only_fast_B=df2emd(word2vec_model_fast, N_rewiews, "review_body")
embed_only_fast_HP=df2emd(word2vec_model_fast, N_rewiews, "review_headline")


embed_combined=embed_only_fast_B.join(embed_only_fast_HP, lsuffix='_caller', rsuffix='_other')




In [35]:
embed_combined.shape

(20000, 600)

In [36]:
run_SVC(embed_combined,N_rewiews)



0.9416
                 precision    recall  f1-score   support

0 = rating of 1   0.951935  0.930705  0.941200      2511
1 = rating of 5   0.931631  0.952591  0.941994      2489

       accuracy                       0.941600      5000
      macro avg   0.941783  0.941648  0.941597      5000
   weighted avg   0.941827  0.941600  0.941596      5000

[[2337  174]
 [ 118 2371]]


In [37]:
#combine the embeddings with the emotions

embed_emptions_combined=embed_combined.join(df_sample_all_test, lsuffix='_caller', rsuffix='_other')



In [38]:
df_sample_all_test.shape

(20000, 21)

In [39]:
embed_emptions_combined.shape

(20000, 621)

In [40]:
run_SVC(embed_emptions_combined,N_rewiews)

0.9526
                 precision    recall  f1-score   support

0 = rating of 1   0.961820  0.943051  0.952343      2511
1 = rating of 5   0.943656  0.962234  0.952855      2489

       accuracy                       0.952600      5000
      macro avg   0.952738  0.952642  0.952599      5000
   weighted avg   0.952778  0.952600  0.952597      5000

[[2368  143]
 [  94 2395]]


In [41]:
run_AdaBoost(embed_emptions_combined,N_rewiews)

0.9498
                 precision    recall  f1-score   support

0 = rating of 1   0.951639  0.948228  0.949930      2511
1 = rating of 5   0.947958  0.951386  0.949669      2489

       accuracy                       0.949800      5000
      macro avg   0.949799  0.949807  0.949800      5000
   weighted avg   0.949807  0.949800  0.949800      5000

[[2381  130]
 [ 121 2368]]


In [43]:
X_train, X_test, y_train, y_test = train_test_split(embed_emptions_combined, N_rewiews['star_rating'],  random_state=56)



#https://scikit-learn.org/stable/auto_examples/model_selection/plot_grid_search_digits.html
#Grid search tuning

from sklearn.model_selection import GridSearchCV

tuned_parameters =  [ {"kernel": ["rbf"], "gamma": [1e-3, 1e-4], "C": [1, 10, 100, 1000]}, {"kernel": ["linear"], "C": [1, 10, 100, 1000]},  ]  
    
scores = ["precision", "recall"]

for score in scores:
    print("# Tuning hyper-parameters for %s" % score)
    print()

    clf = GridSearchCV(SVC(), tuned_parameters, scoring="%s_macro" % score)
    clf.fit(X_train, y_train)

    print("Best parameters set found on Train set:")
    print()
    print(clf.best_params_)
    print()
    print("Grid scores on Train set:")
    print()
    means = clf.cv_results_["mean_test_score"]
    stds = clf.cv_results_["std_test_score"]
    for mean, std, params in zip(means, stds, clf.cv_results_["params"]):
        print("%0.3f (+/-%0.03f) for %r" % (mean, std * 2, params))
    print()

    print("Report for y_true and y_pred:")
    print()
    y_true, y_pred = y_test, clf.predict(X_test)
    print(classification_report(y_true, y_pred))
    print()

# Tuning hyper-parameters for precision

Best parameters set found on Train set:

{'C': 1000, 'gamma': 0.001, 'kernel': 'rbf'}

Grid scores on Train set:

0.930 (+/-0.009) for {'C': 1, 'gamma': 0.001, 'kernel': 'rbf'}
0.901 (+/-0.007) for {'C': 1, 'gamma': 0.0001, 'kernel': 'rbf'}
0.943 (+/-0.009) for {'C': 10, 'gamma': 0.001, 'kernel': 'rbf'}
0.930 (+/-0.009) for {'C': 10, 'gamma': 0.0001, 'kernel': 'rbf'}
0.949 (+/-0.011) for {'C': 100, 'gamma': 0.001, 'kernel': 'rbf'}
0.943 (+/-0.008) for {'C': 100, 'gamma': 0.0001, 'kernel': 'rbf'}
0.952 (+/-0.009) for {'C': 1000, 'gamma': 0.001, 'kernel': 'rbf'}
0.949 (+/-0.010) for {'C': 1000, 'gamma': 0.0001, 'kernel': 'rbf'}
0.951 (+/-0.010) for {'C': 1, 'kernel': 'linear'}
0.948 (+/-0.011) for {'C': 10, 'kernel': 'linear'}
0.944 (+/-0.010) for {'C': 100, 'kernel': 'linear'}
0.943 (+/-0.009) for {'C': 1000, 'kernel': 'linear'}

Report for y_true and y_pred:

              precision    recall  f1-score   support

           1       0.96      0.9

In [51]:
#https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeClassifier.html#sklearn.tree.DecisionTreeClassifier
#REF :https://stackoverflow.com/questions/32210569/using-gridsearchcv-with-adaboost-and-decisiontreeclassifier


X_train, X_test, y_train, y_test = train_test_split(embed_emptions_combined, N_rewiews['star_rating'],  random_state=56)



from sklearn.tree import DecisionTreeClassifier
#from sklearn.grid_search import GridSearchCV

Tree = AdaBoostClassifier(base_estimator=DecisionTreeClassifier(), random_state = 11)

parameters = {'base_estimator__max_depth': [1, 2],
              'base_estimator__min_samples_leaf': [1] ,
              'n_estimators': [10],
              'learning_rate': [0.1]}

clf = GridSearchCV(Tree, parameters,verbose=3,scoring='f1',n_jobs=-1)
clf.fit(X_train,y_train)

print("==================")
print(parameters)
print()
print("Report for y_true and y_pred:")
print()
y_true, y_pred = y_test, clf.predict(X_test)
print(classification_report(y_true, y_pred))




Fitting 5 folds for each of 2 candidates, totalling 10 fits
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENI

In [ ]:
#THIS IS TAKING TOO LONG to RUN. ABORTED.

X_train, X_test, y_train, y_test = train_test_split(embed_emptions_combined, N_rewiews['star_rating'],  random_state=56)



#https://scikit-learn.org/stable/auto_examples/model_selection/plot_grid_search_digits.html
#Grid search tuning

from sklearn.model_selection import GridSearchCV
    
    
tuned_parameters = {'base_estimator__max_depth': [10, 20],#, 10, 20],
              'base_estimator__min_samples_leaf': [1, 10] ,
              'n_estimators': [ 10],#,50,],
              'learning_rate': [  0.1], 
             #"base_estimator__criterion" : ["gini", "entropy"],
              #             "base_estimator__splitter" :   ["best", "random"]
                   }

Tree = AdaBoostClassifier(base_estimator=DecisionTreeClassifier(), random_state = 11)



scores = ["precision", "recall"]

for score in scores:
    print("# Tuning hyper-parameters for %s" % score)
    print()

    clf = GridSearchCV(Tree, tuned_parameters, scoring="%s_macro" % score)
    clf.fit(X_train, y_train)

    print("Best parameters set found on Train set:")
    print()
    print(clf.best_params_)
    print()
    print("Grid scores on Train set:")
    print()
    means = clf.cv_results_["mean_test_score"]
    stds = clf.cv_results_["std_test_score"]
    for mean, std, params in zip(means, stds, clf.cv_results_["params"]):
        print("%0.3f (+/-%0.03f) for %r" % (mean, std * 2, params))
    print()

    print("Report for y_true and y_pred:")
    print()
    y_true, y_pred = y_test, clf.predict(X_test)
    print(classification_report(y_true, y_pred))
    print()

In [61]:
#ADD test_size=0.3
X_train, X_test, y_train, y_test = train_test_split(embed_emptions_combined, N_rewiews['star_rating'],  random_state=56, test_size=0.3)



#https://scikit-learn.org/stable/auto_examples/model_selection/plot_grid_search_digits.html
#Grid search tuning

from sklearn.model_selection import GridSearchCV

tuned_parameters =  [ {"kernel": ["rbf"], "gamma": [1e-3, 1e-4], "C": [1, 10, 100, 1000]}, {"kernel": ["linear"], "C": [1, 10, 100, 1000]},  ]  
    
scores = ["precision", "recall"]

for score in scores:
    print("# Tuning hyper-parameters for %s" % score)
    print()

    clf = GridSearchCV(SVC(), tuned_parameters, scoring="%s_macro" % score)
    clf.fit(X_train, y_train)

    print("Best parameters set found on Train set:")
    print()
    print(clf.best_params_)
    print()
    print("Grid scores on Train set:")
    print()
    means = clf.cv_results_["mean_test_score"]
    stds = clf.cv_results_["std_test_score"]
    for mean, std, params in zip(means, stds, clf.cv_results_["params"]):
        print("%0.3f (+/-%0.03f) for %r" % (mean, std * 2, params))
    print()

    print("Report for y_true and y_pred:")
    print()
    y_true, y_pred = y_test, clf.predict(X_test)
    print(classification_report(y_true, y_pred))
    print()

# Tuning hyper-parameters for precision

Best parameters set found on Train set:

{'C': 1000, 'gamma': 0.001, 'kernel': 'rbf'}

Grid scores on Train set:

0.931 (+/-0.012) for {'C': 1, 'gamma': 0.001, 'kernel': 'rbf'}
0.897 (+/-0.011) for {'C': 1, 'gamma': 0.0001, 'kernel': 'rbf'}
0.943 (+/-0.014) for {'C': 10, 'gamma': 0.001, 'kernel': 'rbf'}
0.931 (+/-0.013) for {'C': 10, 'gamma': 0.0001, 'kernel': 'rbf'}
0.951 (+/-0.010) for {'C': 100, 'gamma': 0.001, 'kernel': 'rbf'}
0.943 (+/-0.013) for {'C': 100, 'gamma': 0.0001, 'kernel': 'rbf'}
0.953 (+/-0.012) for {'C': 1000, 'gamma': 0.001, 'kernel': 'rbf'}
0.950 (+/-0.010) for {'C': 1000, 'gamma': 0.0001, 'kernel': 'rbf'}
0.952 (+/-0.010) for {'C': 1, 'kernel': 'linear'}
0.948 (+/-0.010) for {'C': 10, 'kernel': 'linear'}
0.943 (+/-0.009) for {'C': 100, 'kernel': 'linear'}
0.942 (+/-0.008) for {'C': 1000, 'kernel': 'linear'}

Report for y_true and y_pred:

              precision    recall  f1-score   support

           1       0.96      0.9

In [96]:
N_rewiews.shape

(20000, 38)

In [99]:
N_rewiews.star_rating.value_counts()



1    10000
5    10000
Name: star_rating, dtype: int64

In [97]:
#CROSS PRODUCT ASSESSMENT

#Mobile Electronics Dataset:

import numpy as np
import pandas as pd
fileMobile='amazon_reviews_us_Mobile_Electronics_v1_00.tsv'
df_Mobile=pd.read_csv(fileMobile, sep="\t", header=0, on_bad_lines='skip')
df_Mobile=df_Mobile.dropna(subset=['review_headline', 'review_body', 'star_rating'])



In [98]:
df_Mobile.star_rating.value_counts()

5.0    52196
4.0    18063
1.0    17571
3.0     9719
2.0     7298
Name: star_rating, dtype: int64

In [100]:
X_train, X_test, y_train, y_test = train_test_split(embed_emptions_combined, N_rewiews['star_rating'],  random_state=56, test_size=0.3)



In [101]:
y_test.value_counts()

1    3012
5    2988
Name: star_rating, dtype: int64

In [124]:
#so if we want to apply the  model trained on the Elecrtonics data to the Mobile data then we need
# 3012 of ratings 1 and 2988 of ratings of 5

  
N_rewiews11_Mobile=df_Mobile.loc[df_Mobile['star_rating'] == 1].sample(3012, replace=False, random_state=19300)
N_rewiews22_Mobile=df_Mobile.loc[df_Mobile['star_rating'] == 5].sample(2988, replace=False, random_state=19300)

In [125]:
samplesize= 2988+ 3012

N_Mobile = N_rewiews11_Mobile.append(N_rewiews22_Mobile)

N_Mobile = N_Mobile.reset_index()

/var/folders/ds/k83592y50w34wqwchx8qldph0000gn/T/ipykernel_5253/1866399312.py:3: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  N_Mobile = N_rewiews11_Mobile.append(N_rewiews22_Mobile)


In [126]:
N_Mobile.star_rating.value_counts()

1.0    3012
5.0    2988
Name: star_rating, dtype: int64

In [127]:
N_Mobile =text_process2(N_Mobile,'review_headline')

emotion_roberta(N_Mobile, 'review_body')
emotion_roberta(N_Mobile, 'review_headline')
emotion_roberta(N_Mobile, 'review_headline_processed')










 text_process:   We are at i= 0

 emotion_roberta:   We are at i= 0

 emotion_roberta:   We are at i= 1000

 emotion_roberta:   We are at i= 2000

 emotion_roberta:   We are at i= 3000

 emotion_roberta:   We are at i= 4000

 emotion_roberta:   We are at i= 5000

 emotion_roberta:   We are at i= 0

 emotion_roberta:   We are at i= 1000

 emotion_roberta:   We are at i= 2000

 emotion_roberta:   We are at i= 3000

 emotion_roberta:   We are at i= 4000

 emotion_roberta:   We are at i= 5000

 emotion_roberta:   We are at i= 0

 emotion_roberta:   We are at i= 1000

 emotion_roberta:   We are at i= 2000

 emotion_roberta:   We are at i= 3000

 emotion_roberta:   We are at i= 4000

 emotion_roberta:   We are at i= 5000


,index,marketplace,customer_id,review_id,product_id,product_parent,product_title,product_category,star_rating,helpful_votes,...,roberta_review_headlineneutral,roberta_review_headlinesadness,roberta_review_headlinesurprise,roberta_review_headline_processedanger,roberta_review_headline_processeddisgust,roberta_review_headline_processedfear,roberta_review_headline_processedjoy,roberta_review_headline_processedneutral,roberta_review_headline_processedsadness,roberta_review_headline_processedsurprise
0,79966,US,38616881,R1RRPU6SGLK8DV,B002PZO5BS,507211424,iPod Charger Kit for iPod Nano 5th Generation ...,Mobile_Electronics,1.0,0.0,...,0.067874,0.155644,0.009005,0.153603,0.459098,0.005741,0.002248,0.072508,0.294841,0.011960
1,60657,US,21013179,R280KO0FVT25X0,B004VQFPCC,881499325,BIRUGEAR White USB Home Travel Charger Adapter...,Mobile_Electronics,1.0,0.0,...,0.046010,0.916178,0.005214,0.012712,0.005623,0.010888,0.002687,0.146194,0.735294,0.086602
2,46730,US,29563212,RJKQ2R60ZO9T4,B00AE6L022,432598777,Patuoxun Remote Speaker Microphone for Wouxun ...,Mobile_Electronics,1.0,0.0,...,0.040310,0.014100,0.003292,0.250907,0.729412,0.002760,0.000822,0.007567,0.005402,0.003131
3,85494,US,33186437,RQJ3KAB9TVK3K,B004FT5OIA,746897823,Goal0 Ranger 350 Kit with 1 x Ranger 350 Batte...,Mobile_Electronics,1.0,19.0,...,0.159936,0.121146,0.164743,0.046657,0.016138,0.057290,0.003431,0.294370,0.502200,0.079914
4,48911,US,10108509,R2ELU9R16620BC,B00368NZUK,526238605,"M-Edge Platform Jacket for nook, Jade Green",Mobile_Electronics,1.0,0.0,...,0.730686,0.167808,0.023834,0.009329,0.009293,0.011646,0.005706,0.340784,0.608588,0.014653
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5995,82630,US,18378335,R33DWMFTFX3VPW,B003PSA5BS,61856304,Hifonics Hfi 100.4 4 Channel 800 Watt RMS Ampl...,Mobile_Electronics,5.0,1.0,...,0.896507,0.028632,0.043109,0.007390,0.007510,0.005197,0.011655,0.896507,0.028632,0.043109
5996,66261,US,44761840,R300UJR3EIE20L,B007MJD5I6,376499855,eForCity Leather Case for Barnes and Noble Noo...,Mobile_Electronics,5.0,0.0,...,0.894495,0.005965,0.019058,0.040712,0.020193,0.003471,0.093332,0.794356,0.018289,0.029647
5997,42592,US,10263041,R3RHQ77J9OLRHG,B0085HC2V0,183954191,Motorola TX500 Universal Bluetooth In-Car Spea...,Mobile_Electronics,5.0,0.0,...,0.150621,0.005957,0.056475,0.010750,0.003309,0.004194,0.279321,0.602579,0.023525,0.076320
5998,95232,US,28300190,R2H87SOO18GQWG,B000V5TV22,152954642,USGlobalSat BU-353 WaterProof WAAS Enabled USB...,Mobile_Electronics,5.0,0.0,...,0.154996,0.011903,0.528348,0.016635,0.000941,0.497486,0.032667,0.034613,0.013575,0.404082


In [128]:
N_Mobile.shape

(6000, 38)

In [129]:
df_sample_all_N_Mobile = N_Mobile[ Roberta_Body+Roberta_Head+Roberta_Head_Processed ] 


In [130]:
embed_only_fast_B_Mobile =df2emd(word2vec_model_fast, N_Mobile, "review_body")
embed_only_fast_HP_Mobile =df2emd(word2vec_model_fast, N_Mobile, "review_headline")

embed_combined_Mobile = embed_only_fast_B_Mobile.join(embed_only_fast_HP_Mobile, lsuffix='_caller', rsuffix='_other')



In [131]:
#combine the embeddings with the emotions

embed_emotions_combined_Mobile=embed_combined_Mobile.join(df_sample_all_N_Mobile, lsuffix='_caller', rsuffix='_other')

embed_emotions_combined_Mobile



,0_caller,1_caller,2_caller,3_caller,4_caller,5_caller,6_caller,7_caller,8_caller,9_caller,...,roberta_review_headlineneutral,roberta_review_headlinesadness,roberta_review_headlinesurprise,roberta_review_headline_processedanger,roberta_review_headline_processeddisgust,roberta_review_headline_processedfear,roberta_review_headline_processedjoy,roberta_review_headline_processedneutral,roberta_review_headline_processedsadness,roberta_review_headline_processedsurprise
0,-0.051547,-0.022103,-0.029978,0.003303,-0.011866,0.026504,-0.026559,0.053534,0.017329,-0.018252,...,0.067874,0.155644,0.009005,0.153603,0.459098,0.005741,0.002248,0.072508,0.294841,0.011960
1,-0.037820,-0.037714,-0.046389,0.024649,0.028789,0.071389,-0.062660,0.034651,0.076397,0.006517,...,0.046010,0.916178,0.005214,0.012712,0.005623,0.010888,0.002687,0.146194,0.735294,0.086602
2,-0.025788,-0.015939,-0.008953,-0.025320,-0.038282,0.038329,-0.020335,0.032635,0.043565,-0.005794,...,0.040310,0.014100,0.003292,0.250907,0.729412,0.002760,0.000822,0.007567,0.005402,0.003131
3,-0.018252,-0.065886,-0.016570,-0.026131,-0.002698,0.024401,-0.014035,0.032453,0.037664,-0.012297,...,0.159936,0.121146,0.164743,0.046657,0.016138,0.057290,0.003431,0.294370,0.502200,0.079914
4,-0.052203,-0.029647,-0.032833,-0.011136,-0.041931,0.038131,-0.045447,0.041117,0.032322,0.016244,...,0.730686,0.167808,0.023834,0.009329,0.009293,0.011646,0.005706,0.340784,0.608588,0.014653
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5995,-0.071471,-0.024838,-0.126252,0.037729,-0.057667,0.008400,-0.019205,0.049643,0.027419,-0.077395,...,0.896507,0.028632,0.043109,0.007390,0.007510,0.005197,0.011655,0.896507,0.028632,0.043109
5996,-0.117104,0.017860,-0.038472,-0.016560,-0.026956,0.039956,-0.003304,0.093396,-0.051952,-0.046048,...,0.894495,0.005965,0.019058,0.040712,0.020193,0.003471,0.093332,0.794356,0.018289,0.029647
5997,-0.052736,-0.110988,0.003442,-0.004606,-0.025548,0.050306,-0.035370,-0.016773,0.009927,-0.040036,...,0.150621,0.005957,0.056475,0.010750,0.003309,0.004194,0.279321,0.602579,0.023525,0.076320
5998,-0.037526,0.030594,-0.011617,0.019427,0.012391,0.057685,0.009097,0.026408,0.054235,0.008091,...,0.154996,0.011903,0.528348,0.016635,0.000941,0.497486,0.032667,0.034613,0.013575,0.404082


In [132]:
embed_emotions_combined_Mobile.shape

(6000, 621)

In [133]:
X_test=embed_emotions_combined_Mobile
y_test=N_Mobile['star_rating']

In [134]:
N_Mobile['star_rating'].value_counts()

1.0    3012
5.0    2988
Name: star_rating, dtype: int64

In [135]:
X_test

,0_caller,1_caller,2_caller,3_caller,4_caller,5_caller,6_caller,7_caller,8_caller,9_caller,...,roberta_review_headlineneutral,roberta_review_headlinesadness,roberta_review_headlinesurprise,roberta_review_headline_processedanger,roberta_review_headline_processeddisgust,roberta_review_headline_processedfear,roberta_review_headline_processedjoy,roberta_review_headline_processedneutral,roberta_review_headline_processedsadness,roberta_review_headline_processedsurprise
0,-0.051547,-0.022103,-0.029978,0.003303,-0.011866,0.026504,-0.026559,0.053534,0.017329,-0.018252,...,0.067874,0.155644,0.009005,0.153603,0.459098,0.005741,0.002248,0.072508,0.294841,0.011960
1,-0.037820,-0.037714,-0.046389,0.024649,0.028789,0.071389,-0.062660,0.034651,0.076397,0.006517,...,0.046010,0.916178,0.005214,0.012712,0.005623,0.010888,0.002687,0.146194,0.735294,0.086602
2,-0.025788,-0.015939,-0.008953,-0.025320,-0.038282,0.038329,-0.020335,0.032635,0.043565,-0.005794,...,0.040310,0.014100,0.003292,0.250907,0.729412,0.002760,0.000822,0.007567,0.005402,0.003131
3,-0.018252,-0.065886,-0.016570,-0.026131,-0.002698,0.024401,-0.014035,0.032453,0.037664,-0.012297,...,0.159936,0.121146,0.164743,0.046657,0.016138,0.057290,0.003431,0.294370,0.502200,0.079914
4,-0.052203,-0.029647,-0.032833,-0.011136,-0.041931,0.038131,-0.045447,0.041117,0.032322,0.016244,...,0.730686,0.167808,0.023834,0.009329,0.009293,0.011646,0.005706,0.340784,0.608588,0.014653
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5995,-0.071471,-0.024838,-0.126252,0.037729,-0.057667,0.008400,-0.019205,0.049643,0.027419,-0.077395,...,0.896507,0.028632,0.043109,0.007390,0.007510,0.005197,0.011655,0.896507,0.028632,0.043109
5996,-0.117104,0.017860,-0.038472,-0.016560,-0.026956,0.039956,-0.003304,0.093396,-0.051952,-0.046048,...,0.894495,0.005965,0.019058,0.040712,0.020193,0.003471,0.093332,0.794356,0.018289,0.029647
5997,-0.052736,-0.110988,0.003442,-0.004606,-0.025548,0.050306,-0.035370,-0.016773,0.009927,-0.040036,...,0.150621,0.005957,0.056475,0.010750,0.003309,0.004194,0.279321,0.602579,0.023525,0.076320
5998,-0.037526,0.030594,-0.011617,0.019427,0.012391,0.057685,0.009097,0.026408,0.054235,0.008091,...,0.154996,0.011903,0.528348,0.016635,0.000941,0.497486,0.032667,0.034613,0.013575,0.404082


In [136]:
X_train

,0_caller,1_caller,2_caller,3_caller,4_caller,5_caller,6_caller,7_caller,8_caller,9_caller,...,roberta_review_headlineneutral,roberta_review_headlinesadness,roberta_review_headlinesurprise,roberta_review_headline_processedanger,roberta_review_headline_processeddisgust,roberta_review_headline_processedfear,roberta_review_headline_processedjoy,roberta_review_headline_processedneutral,roberta_review_headline_processedsadness,roberta_review_headline_processedsurprise
7363,-0.071477,-0.019920,-0.009947,-0.011210,-0.018612,0.043074,-0.017922,0.027727,0.031480,0.004832,...,0.009592,0.018135,0.909591,0.003441,0.001876,0.016925,0.004683,0.004892,0.004661,0.963523
12320,-0.153167,-0.119583,-0.082850,0.068517,-0.221617,0.043917,-0.086983,0.021550,-0.030583,-0.056833,...,0.874900,0.009004,0.035351,0.017302,0.005385,0.002486,0.472787,0.424965,0.012054,0.065021
16809,-0.116671,0.041204,0.014710,0.071717,0.030388,0.069340,-0.035600,0.000356,0.075198,0.106450,...,0.704509,0.019287,0.188875,0.017816,0.005648,0.033493,0.007669,0.355443,0.024164,0.555767
18145,-0.049777,0.035276,-0.016835,-0.041458,-0.021500,-0.013172,-0.009153,0.020551,0.039782,-0.012552,...,0.474228,0.011277,0.075464,0.005690,0.014081,0.015997,0.164111,0.245017,0.015227,0.539877
8692,-0.062091,-0.046279,-0.028053,-0.012519,-0.023161,-0.000878,-0.025980,0.060577,0.064909,0.010511,...,0.044635,0.710779,0.024365,0.240469,0.023359,0.014132,0.003097,0.038949,0.651602,0.028393
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9338,-0.054450,-0.070981,-0.014473,-0.068249,0.007750,0.017211,-0.028519,-0.028934,0.075560,0.037559,...,0.419256,0.135263,0.111239,0.018949,0.006030,0.014148,0.004203,0.361692,0.154242,0.440736
13730,-0.049592,0.027235,-0.023601,-0.006246,-0.030634,0.027540,-0.007796,0.031728,0.040165,0.004577,...,0.088796,0.015193,0.532414,0.015950,0.026261,0.014244,0.021960,0.793885,0.040253,0.087447
3264,-0.083210,-0.010530,-0.018235,0.019595,-0.017625,0.038344,0.009862,0.005537,0.039368,-0.015014,...,0.113629,0.833209,0.007758,0.003796,0.005327,0.002559,0.011609,0.797812,0.090652,0.088245
399,-0.076713,0.115987,-0.014225,-0.035900,0.053700,0.023212,0.010350,0.083800,-0.138912,-0.017763,...,0.858583,0.023295,0.037349,0.007135,0.003841,0.003411,0.024708,0.438367,0.356512,0.166027


In [112]:
#NOw TRAIN on the ELECRTONIcS X_train  y_train but test on the X_test and y_test from the above line

In [137]:
target_names = ['0 = rating of 1',  '1 = rating of 5'] # 0 = negative, 4 = positive


clf = make_pipeline(StandardScaler(), SVC( C=1000, gamma= 0.001, kernel= 'rbf'))
clf.fit(X_train, y_train)
y_pred=clf.predict(X_test)

clf.score(X_test, y_test)
y_test.value_counts()
print(clf.score(X_test, y_test))

print(classification_report(y_test, y_pred, target_names=target_names, digits=6))

print(confusion_matrix(y_test, y_pred))






0.9451666666666667
                 precision    recall  f1-score   support

0 = rating of 1   0.947913  0.942563  0.945231      3012
1 = rating of 5   0.942429  0.947791  0.945103      2988

       accuracy                       0.945167      6000
      macro avg   0.945171  0.945177  0.945167      6000
   weighted avg   0.945182  0.945167  0.945167      6000

[[2839  173]
 [ 156 2832]]


In [138]:
#TRY AdaBoostClassifier

clf = AdaBoostClassifier(n_estimators=50, random_state=156)
    
clf.fit(X_train, y_train)
y_pred=clf.predict(X_test)

print(clf.score(X_test, y_test))

clf.score(X_test, y_test)
target_names = ['0/NEG = ratings of 1,2',  '1/POS = ratings of 4,5'] #e


print(classification_report(y_test, y_pred, target_names=target_names, digits=6))

print(confusion_matrix(y_test, y_pred))


0.9388333333333333
                        precision    recall  f1-score   support

0/NEG = ratings of 1,2   0.946942  0.930279  0.938536      3012
1/POS = ratings of 4,5   0.930944  0.947456  0.939128      2988

              accuracy                       0.938833      6000
             macro avg   0.938943  0.938868  0.938832      6000
          weighted avg   0.938975  0.938833  0.938831      6000

[[2802  210]
 [ 157 2831]]


In [139]:
#CROSS PRODUCT ASSESSMENT

#Mobile Electronics Dataset:

def cross_prod(filename, n_1, n_5): #n_1 number of ratings of 1 to sample

    #fileMobile='amazon_reviews_us_Mobile_Electronics_v1_00.tsv'
    df_Mobile=pd.read_csv(filename, sep="\t", header=0, on_bad_lines='skip')
    df_Mobile=df_Mobile.dropna(subset=['review_headline', 'review_body', 'star_rating'])


    print(df_Mobile.star_rating.value_counts())

    N_rewiews11_Mobile=df_Mobile.loc[df_Mobile['star_rating'] == 1].sample(n_1, replace=False, random_state=19300)
    N_rewiews22_Mobile=df_Mobile.loc[df_Mobile['star_rating'] == 5].sample(n_5, replace=False, random_state=19300)

    samplesize= n_1+ n_5

    N_Mobile = N_rewiews11_Mobile.append(N_rewiews22_Mobile)

    N_Mobile = N_Mobile.reset_index()

    print(N_Mobile.star_rating.value_counts() ) 


    N_Mobile =text_process2(N_Mobile,'review_headline')

    emotion_roberta(N_Mobile, 'review_body')
    emotion_roberta(N_Mobile, 'review_headline')
    emotion_roberta(N_Mobile, 'review_headline_processed')



    df_sample_all_N_Mobile = N_Mobile[ Roberta_Body+Roberta_Head+Roberta_Head_Processed ] 

    embed_only_fast_B_Mobile =df2emd(word2vec_model_fast, N_Mobile, "review_body")
    embed_only_fast_HP_Mobile =df2emd(word2vec_model_fast, N_Mobile, "review_headline")

    embed_combined_Mobile = embed_only_fast_B_Mobile.join(embed_only_fast_HP_Mobile, lsuffix='_caller', rsuffix='_other')


    #combine the embeddings with the emotions

    embed_emotions_combined_Mobile=embed_combined_Mobile.join(df_sample_all_N_Mobile, lsuffix='_caller', rsuffix='_other')

    embed_emotions_combined_Mobile


    X_test=embed_emotions_combined_Mobile
    y_test=N_Mobile['star_rating']


    return X_test, y_test






In [142]:


filename="amazon_reviews_us_Digital_Video_Download_v1_00.tsv"

n_1=3012
n_5=2988
  
X_test, y_test=cross_prod(filename, n_1, n_5) #n_1 number of ratings of 1 to sample



print('===amazon_reviews_us_Luggage_v1_00==')
filename='amazon_reviews_us_Luggage_v1_00.tsv'
X_test2, y_test2=cross_prod(filename, n_1, n_5) #n_1 number of ratings of 1 to sample




5    2410475
4     756399
3     345877
1     289727
2     195764
Name: star_rating, dtype: int64


/var/folders/ds/k83592y50w34wqwchx8qldph0000gn/T/ipykernel_5253/3031689699.py:19: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  N_Mobile = N_rewiews11_Mobile.append(N_rewiews22_Mobile)


1    3012
5    2988
Name: star_rating, dtype: int64

 text_process:   We are at i= 0

 emotion_roberta:   We are at i= 0

 emotion_roberta:   We are at i= 1000

 emotion_roberta:   We are at i= 2000

 emotion_roberta:   We are at i= 3000

 emotion_roberta:   We are at i= 4000

 emotion_roberta:   We are at i= 5000

 emotion_roberta:   We are at i= 0

 emotion_roberta:   We are at i= 1000

 emotion_roberta:   We are at i= 2000

 emotion_roberta:   We are at i= 3000

 emotion_roberta:   We are at i= 4000

 emotion_roberta:   We are at i= 5000

 emotion_roberta:   We are at i= 0

 emotion_roberta:   We are at i= 1000

 emotion_roberta:   We are at i= 2000

 emotion_roberta:   We are at i= 3000

 emotion_roberta:   We are at i= 4000

 emotion_roberta:   We are at i= 5000
===amazon_reviews_us_Luggage_v1_00==
5.0    216366
4.0     61388
3.0     27833
1.0     24993
2.0     17854
Name: star_rating, dtype: int64
1.0    3012
5.0    2988
Name: star_rating, dtype: int64

 text_process:   We are at

/var/folders/ds/k83592y50w34wqwchx8qldph0000gn/T/ipykernel_5253/3031689699.py:19: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  N_Mobile = N_rewiews11_Mobile.append(N_rewiews22_Mobile)



 emotion_roberta:   We are at i= 0

 emotion_roberta:   We are at i= 1000

 emotion_roberta:   We are at i= 2000

 emotion_roberta:   We are at i= 3000

 emotion_roberta:   We are at i= 4000

 emotion_roberta:   We are at i= 5000

 emotion_roberta:   We are at i= 0

 emotion_roberta:   We are at i= 1000

 emotion_roberta:   We are at i= 2000

 emotion_roberta:   We are at i= 3000

 emotion_roberta:   We are at i= 4000

 emotion_roberta:   We are at i= 5000

 emotion_roberta:   We are at i= 0

 emotion_roberta:   We are at i= 1000

 emotion_roberta:   We are at i= 2000

 emotion_roberta:   We are at i= 3000

 emotion_roberta:   We are at i= 4000

 emotion_roberta:   We are at i= 5000


In [144]:
#KEEP the TRAIN AS BEFORE

target_names = ['0 = rating of 1',  '1 = rating of 5'] # 0 = negative, 4 = positive



print()
print("===============SVM=amazon_reviews_us_Digital_Video_Download_v1_00==============")
clf = make_pipeline(StandardScaler(), SVC( C=1000, gamma= 0.001, kernel= 'rbf'))
clf.fit(X_train, y_train)

y_pred=clf.predict(X_test)

y_test.value_counts()
print(clf.score(X_test, y_test))
print(classification_report(y_test, y_pred, target_names=target_names, digits=6))
print(confusion_matrix(y_test, y_pred))


print("===============SVM=amazon_reviews_us_Luggage_v1_00==============")
y_pred=clf.predict(X_test2)

y_test.value_counts()
print(clf.score(X_test2, y_test2))
print(classification_report(y_test2, y_pred, target_names=target_names, digits=6))
print(confusion_matrix(y_test2, y_pred))




#TRY AdaBoostClassifier

print()
print("===============AdaBoostClassifier=amazon_reviews_us_Digital_Video_Download_v1_00===============")
clf = AdaBoostClassifier(n_estimators=50, random_state=156)    
clf.fit(X_train, y_train)

y_pred=clf.predict(X_test)

clf.score(X_test, y_test)
print(clf.score(X_test, y_test))
print(classification_report(y_test, y_pred, target_names=target_names, digits=6))
print(confusion_matrix(y_test, y_pred))


print("===============AdaBoostClassifier=amazon_reviews_us_Luggage_v1_00===============")
y_pred=clf.predict(X_test2)

y_test.value_counts()
print(clf.score(X_test2, y_test2))
print(classification_report(y_test2, y_pred, target_names=target_names, digits=6))
print(confusion_matrix(y_test2, y_pred))







===============SVM=amazon_reviews_us_Digital_Video_Download_v1_00==============
0.9075
                 precision    recall  f1-score   support

0 = rating of 1   0.933946  0.877822  0.905015      3012
1 = rating of 5   0.883875  0.937416  0.909859      2988

       accuracy                       0.907500      6000
      macro avg   0.908910  0.907619  0.907437      6000
   weighted avg   0.909010  0.907500  0.907427      6000

[[2644  368]
 [ 187 2801]]
===============SVM=amazon_reviews_us_Luggage_v1_00==============
0.9365
                 precision    recall  f1-score   support

0 = rating of 1   0.937479  0.935923  0.936700      3012
1 = rating of 5   0.935516  0.937082  0.936298      2988

       accuracy                       0.936500      6000
      macro avg   0.936498  0.936502  0.936499      6000
   weighted avg   0.936502  0.936500  0.936500      6000

[[2819  193]
 [ 188 2800]]

===============AdaBoostClassifier=amazon_reviews_us_Digital_Video_Download_v1_00===============

In [ ]:
#ADDING 2's and 4's but still keeping it as two classes (USING THE ELECTRONICS DATASET)

In [76]:
n_samples=500

N_rewiews11=df.loc[df['star_rating'] == 1].sample(n_samples, replace=False, random_state=19300)
N_rewiews22=df.loc[df['star_rating'] == 2].sample(n_samples, replace=False, random_state=19300)
N_rewiew44=df.loc[df['star_rating'] == 4].sample(n_samples, replace=False, random_state=19300)
N_rewiew55=df.loc[df['star_rating'] == 5 ].sample(n_samples, replace=False, random_state=19300)

samplesize=n_samples*4

N_rewiews11 = N_rewiews11.append(N_rewiews22)

N_rewiews11 = N_rewiews11.append(N_rewiew44)

N_rewiews11 = N_rewiews11.append(N_rewiew55)


/var/folders/ds/k83592y50w34wqwchx8qldph0000gn/T/ipykernel_5253/3493751854.py:10: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  N_rewiews11 = N_rewiews11.append(N_rewiews22)
/var/folders/ds/k83592y50w34wqwchx8qldph0000gn/T/ipykernel_5253/3493751854.py:12: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  N_rewiews11 = N_rewiews11.append(N_rewiew44)
/var/folders/ds/k83592y50w34wqwchx8qldph0000gn/T/ipykernel_5253/3493751854.py:14: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  N_rewiews11 = N_rewiews11.append(N_rewiew55)


In [77]:
N_rewiews11.star_rating.value_counts()

1    500
2    500
4    500
5    500
Name: star_rating, dtype: int64

In [78]:
N_rewiews11=N_rewiews11.reset_index()

N_rewiews11 =text_process2(N_rewiews11,'review_headline')


 text_process:   We are at i= 0


In [80]:
N_rewiews11.shape

(2000, 17)

In [84]:
emotion_roberta(N_rewiews11, 'review_body')
emotion_roberta(N_rewiews11, 'review_headline')
emotion_roberta(N_rewiews11, 'review_headline_processed')




 emotion_roberta:   We are at i= 0

 emotion_roberta:   We are at i= 1000

 emotion_roberta:   We are at i= 0

 emotion_roberta:   We are at i= 1000

 emotion_roberta:   We are at i= 0

 emotion_roberta:   We are at i= 1000


,index,marketplace,customer_id,review_id,product_id,product_parent,product_title,product_category,star_rating,helpful_votes,...,roberta_review_headlineneutral,roberta_review_headlinesadness,roberta_review_headlinesurprise,roberta_review_headline_processedanger,roberta_review_headline_processeddisgust,roberta_review_headline_processedfear,roberta_review_headline_processedjoy,roberta_review_headline_processedneutral,roberta_review_headline_processedsadness,roberta_review_headline_processedsurprise
0,1032877,US,13218409,R33SGQQS0MXN8G,B00E6BTHEO,815769011,Etekcity® VGA & Component + Audio to HDMI Conv...,Electronics,1,0,...,0.197994,0.695608,0.080722,0.003473,0.001205,0.011497,0.004119,0.052285,0.879278,0.048144
1,1820967,US,20215034,RDJIESAA5VFV8,B003Q9PG24,323900231,XL-2400 Replacement Lamp with Housing for Sony...,Electronics,1,0,...,0.776437,0.045698,0.111845,0.064134,0.104007,0.051363,0.040564,0.549477,0.111690,0.078765
2,325471,US,1729530,R13UOS3VCWVU05,B00LG71JZG,195415550,Apple iPod touch (5th Generation) NEWEST MODEL,Electronics,1,4,...,0.828864,0.046700,0.081617,0.011786,0.007271,0.003675,0.021684,0.536597,0.069132,0.349855
3,2139326,US,29846720,RJM9OYK7L6RWU,B003WJ3T8G,3721878,LEDwholesalers Waterproof Electronic LED Drive...,Electronics,1,0,...,0.689784,0.208149,0.071037,0.045088,0.013269,0.010609,0.007745,0.250111,0.497548,0.175630
4,1122125,US,39043237,R3KBYZ375VURBC,B00COMTYPO,833457988,Hooshion® 3.5mm Mini Portable Stereo Speaker f...,Electronics,1,0,...,0.030704,0.860504,0.049787,0.066537,0.002333,0.854129,0.002023,0.011281,0.055931,0.007764
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1995,1501439,US,46957492,R63DMZKYDK6N,B000C46OCC,282511574,"Bose® Wave® music system multi-CD changer, Gra...",Electronics,5,0,...,0.817682,0.052215,0.079050,0.006699,0.007451,0.003180,0.009484,0.685118,0.081084,0.206984
1996,3019475,US,18820209,R3887KW8FDEQU0,B0009KF2P4,612609751,"Velocity Nylon 600D CD Wallet (264 capacity, B...",Electronics,5,5,...,0.749515,0.010314,0.029561,0.012251,0.004955,0.001894,0.615300,0.302956,0.015795,0.046849
1997,2224614,US,10947218,R4LO3EV2KOH8L,B004GW5PTY,677799256,BlueRigger High Speed HDMI Cable with Ethernet...,Electronics,5,0,...,0.054954,0.177072,0.038042,0.020819,0.006594,0.002452,0.021279,0.522907,0.092586,0.333362
1998,1166262,US,18636344,R3NOQI53Z8ZN3S,B00BBDL0HW,791043040,Audio Technica ATHCKP500BL Sporfit In-ear Head...,Electronics,5,5,...,0.524355,0.010951,0.146404,0.004678,0.003200,0.002015,0.165703,0.479896,0.030009,0.314500


In [87]:
df_sample_all_test_N_rewiews11 = N_rewiews11[ Roberta_Body+Roberta_Head+Roberta_Head_Processed ] 



In [86]:
embed_only_fast_B_N_rewiews11 =df2emd(word2vec_model_fast, N_rewiews11, "review_body")
embed_only_fast_HP_N_rewiews11 =df2emd(word2vec_model_fast, N_rewiews11, "review_headline")

embed_combined_N_rewiews11 = embed_only_fast_B_N_rewiews11.join(embed_only_fast_HP_N_rewiews11, lsuffix='_caller', rsuffix='_other')



In [88]:
#combine the embeddings with the emotions

embed_emptions_combined_N_rewiews11=embed_combined_N_rewiews11.join(df_sample_all_test_N_rewiews11, lsuffix='_caller', rsuffix='_other')

embed_emptions_combined_N_rewiews11


,0_caller,1_caller,2_caller,3_caller,4_caller,5_caller,6_caller,7_caller,8_caller,9_caller,...,roberta_review_headlineneutral,roberta_review_headlinesadness,roberta_review_headlinesurprise,roberta_review_headline_processedanger,roberta_review_headline_processeddisgust,roberta_review_headline_processedfear,roberta_review_headline_processedjoy,roberta_review_headline_processedneutral,roberta_review_headline_processedsadness,roberta_review_headline_processedsurprise
0,-0.058000,-0.072641,-0.039610,-0.012356,-0.021541,0.038849,0.003313,0.067154,0.020180,0.012421,...,0.197994,0.695608,0.080722,0.003473,0.001205,0.011497,0.004119,0.052285,0.879278,0.048144
1,-0.090958,-0.030383,-0.041700,-0.032867,-0.002371,0.026883,0.044996,0.078267,0.013550,-0.069783,...,0.776437,0.045698,0.111845,0.064134,0.104007,0.051363,0.040564,0.549477,0.111690,0.078765
2,-0.054853,0.003929,-0.044767,-0.029856,0.022316,0.051227,-0.041420,0.000411,-0.041336,-0.030858,...,0.828864,0.046700,0.081617,0.011786,0.007271,0.003675,0.021684,0.536597,0.069132,0.349855
3,-0.028142,-0.075340,-0.061240,-0.025189,-0.002368,0.021840,0.016994,-0.005229,0.091365,0.047544,...,0.689784,0.208149,0.071037,0.045088,0.013269,0.010609,0.007745,0.250111,0.497548,0.175630
4,-0.044864,-0.059786,-0.088600,-0.001555,-0.019332,0.037036,-0.008609,0.041736,0.041641,-0.003464,...,0.030704,0.860504,0.049787,0.066537,0.002333,0.854129,0.002023,0.011281,0.055931,0.007764
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1995,-0.054982,-0.087750,0.020536,0.039836,-0.040182,0.056625,-0.013929,0.012296,0.055232,0.009614,...,0.817682,0.052215,0.079050,0.006699,0.007451,0.003180,0.009484,0.685118,0.081084,0.206984
1996,-0.059665,0.011440,-0.009678,-0.006251,-0.041743,0.020511,-0.043059,0.055687,0.025482,-0.016070,...,0.749515,0.010314,0.029561,0.012251,0.004955,0.001894,0.615300,0.302956,0.015795,0.046849
1997,-0.064639,-0.030124,-0.021719,0.021369,-0.017619,0.031456,-0.032248,0.038205,0.065376,-0.015075,...,0.054954,0.177072,0.038042,0.020819,0.006594,0.002452,0.021279,0.522907,0.092586,0.333362
1998,-0.046169,0.012950,-0.005875,-0.018919,-0.048233,0.006628,0.004753,-0.011738,0.043711,0.011205,...,0.524355,0.010951,0.146404,0.004678,0.003200,0.002015,0.165703,0.479896,0.030009,0.314500


In [91]:
#create a column where 1,2 = NEG and 4,5=POS

N_rewiews11['sentiment']=np.where(N_rewiews11['star_rating']>=4, 1, 0)



In [92]:
#AdaBoostClassifier


X_train, X_test, y_train, y_test = train_test_split(embed_emptions_combined_N_rewiews11, N_rewiews11['sentiment'],  random_state=526, test_size=0.3)

clf = AdaBoostClassifier(n_estimators=50, random_state=156)
    
clf.fit(X_train, y_train)
y_pred=clf.predict(X_test)

print(clf.score(X_test, y_test))

clf.score(X_test, y_test)
target_names = ['0/NEG = ratings of 1,2',  '1/POS = ratings of 4,5'] #e


print(classification_report(y_test, y_pred, target_names=target_names, digits=6))

print(confusion_matrix(y_test, y_pred))



0.855
                        precision    recall  f1-score   support

0/NEG = ratings of 1,2   0.835526  0.872852  0.853782       291
1/POS = ratings of 4,5   0.875000  0.838188  0.856198       309

              accuracy                       0.855000       600
             macro avg   0.855263  0.855520  0.854990       600
          weighted avg   0.855855  0.855000  0.855026       600

[[254  37]
 [ 50 259]]


In [93]:
#TWO CLASS but combined ratings 1,2 and 4,5

target_names = ['0/NEG = ratings of 1,2',  '1/POS = ratings of 4,5'] #e


X_train, X_test, y_train, y_test = train_test_split(embed_emptions_combined_N_rewiews11, N_rewiews11['sentiment'],  random_state=526, test_size=0.3)


clf = make_pipeline(StandardScaler(), SVC( C=1000, gamma= 0.001, kernel= 'rbf'))
clf.fit(X_train, y_train)
y_pred=clf.predict(X_test)

clf.score(X_test, y_test)
y_test.value_counts()
print(clf.score(X_test, y_test))

print(classification_report(y_test, y_pred, target_names=target_names, digits=6))

print(confusion_matrix(y_test, y_pred))





0.8566666666666667
                        precision    recall  f1-score   support

0/NEG = ratings of 1,2   0.845118  0.862543  0.853741       291
1/POS = ratings of 4,5   0.867987  0.851133  0.859477       309

              accuracy                       0.856667       600
             macro avg   0.856552  0.856838  0.856609       600
          weighted avg   0.856895  0.856667  0.856695       600

[[251  40]
 [ 46 263]]


In [95]:
#MULTI CLASS TESTING / 4 classes

target_names = ['CLASS 0/NEG = ratings of 1', 'CLASS 1/NEG = ratings of 2' , 'CLASS 2/POS = ratings of 4',  'CLASS 3/POS = ratings of 5'] #e


X_train, X_test, y_train, y_test = train_test_split(embed_emptions_combined_N_rewiews11, N_rewiews11['star_rating'],  random_state=526, test_size=0.3)


clf = make_pipeline(StandardScaler(), SVC( C=1000, gamma= 0.001, kernel= 'rbf'))
clf.fit(X_train, y_train)
y_pred=clf.predict(X_test)

clf.score(X_test, y_test)
y_test.value_counts()
print(clf.score(X_test, y_test))

print(classification_report(y_test, y_pred, target_names=target_names, digits=6))

print(confusion_matrix(y_test, y_pred))






0.6133333333333333
                            precision    recall  f1-score   support

CLASS 0/NEG = ratings of 1   0.712418  0.689873  0.700965       158
CLASS 1/NEG = ratings of 2   0.522581  0.609023  0.562500       133
CLASS 2/POS = ratings of 4   0.511450  0.468531  0.489051       143
CLASS 3/POS = ratings of 5   0.689441  0.668675  0.678899       166

                  accuracy                       0.613333       600
                 macro avg   0.608973  0.609026  0.607854       600
              weighted avg   0.616083  0.613333  0.613661       600

[[109  36   9   4]
 [ 30  81  15   7]
 [  7  30  67  39]
 [  7   8  40 111]]


In [ ]:
#mult-class expectedly produces lower accuracies.